# Taxonomy Clustering

## Loading (Lib + df)

In [ ]:
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))

In [ ]:
import pandas as pd

import numpy as np
from sklearn.metrics import adjusted_rand_score

DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

In [12]:
USE_INTERSECTION = True  # True  → intersect mistral ∩ deepseek (aligned pair)


def load_and_enrich(name):
    return (
        pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / f"{name}.parquet")
        .merge(
            pd.read_parquet(DATA_DIR / "taxonomy" / "per_llm" / "google.parquet")[
                ["bibkey", "google_task_type", "title", "abstract"]
            ],
            on="bibkey",
        )
        .merge(
            pd.read_parquet(
                DATA_DIR / "taxonomy" / "per_llm" / "openai_coarse.parquet"
            )[["bibkey", "openai_coarse_task"]],
            on="bibkey",
        )
    )


mistral_taxonomy = load_and_enrich("mistral_extraction_full")
print(f"Après load+merge — mistral: {len(mistral_taxonomy):,}")

if USE_INTERSECTION:
    deepseek_taxonomy = load_and_enrich("deepseek_extraction")
    print(f"Après load+merge — deepseek: {len(deepseek_taxonomy):,}")

    common = set(mistral_taxonomy.bibkey) & set(deepseek_taxonomy.bibkey)
    mistral_taxonomy = mistral_taxonomy[mistral_taxonomy.bibkey.isin(common)]
    deepseek_taxonomy = deepseek_taxonomy[deepseek_taxonomy.bibkey.isin(common)]
    print(f"Après intersection — {len(common):,} bibkeys")

    for df in (mistral_taxonomy, deepseek_taxonomy):
        df["is_multi_record"] = df.groupby("bibkey")["bibkey"].transform("size") > 1

    single = set(
        mistral_taxonomy.loc[~mistral_taxonomy.is_multi_record, "bibkey"]
    ) & set(deepseek_taxonomy.loc[~deepseek_taxonomy.is_multi_record, "bibkey"])
    mistral_taxonomy = mistral_taxonomy[mistral_taxonomy.bibkey.isin(single)]
    deepseek_taxonomy = deepseek_taxonomy[deepseek_taxonomy.bibkey.isin(single)]
    print(f"Après filtre multi — {len(single):,} bibkeys")

    mistral_taxonomy = mistral_taxonomy.sort_values("bibkey").reset_index(drop=True)
    deepseek_taxonomy = deepseek_taxonomy.sort_values("bibkey").reset_index(drop=True)
    assert (mistral_taxonomy.bibkey.values == deepseek_taxonomy.bibkey.values).all()
    print("Ordre bibkeys aligné ✓")

else:
    mistral_taxonomy["is_multi_record"] = (
        mistral_taxonomy.groupby("bibkey")["bibkey"].transform("size") > 1
    )
    mistral_taxonomy = mistral_taxonomy[~mistral_taxonomy.is_multi_record]
    mistral_taxonomy = mistral_taxonomy.sort_values("bibkey").reset_index(drop=True)
    print(f"Après filtre multi — {len(mistral_taxonomy):,} bibkeys")
    deepseek_taxonomy = None
    embeddings_deepseek = None


Après load+merge — mistral: 352
Après load+merge — deepseek: 350
Après intersection — 299 bibkeys
Après filtre multi — 247 bibkeys
Ordre bibkeys aligné ✓


In [ ]:
if USE_INTERSECTION:
    embeddings_mistral = np.load(
        DATA_DIR / "taxonomy" / "per_llm" / "mistral_embeddings_300.npy"
    )
    embeddings_deepseek = np.load(
        DATA_DIR / "taxonomy" / "per_llm" / "deepseek_embeddings_300.npy"
    )
else:
    embeddings_mistral = np.load(
        DATA_DIR / "taxonomy" / "per_llm" / "mistral_embeddings_full.npy"
    )

## 1. Embeddings

In [13]:
from src.taxonomy.embedding import compute_embeddings

embeddings_mistral = compute_embeddings(
    mistral_taxonomy["paraphrase_task"].astype(str).tolist()
)
np.save(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral_embeddings.npy", embeddings_mistral
)

# embeddings_deepseek = compute_embeddings(
#     deepseek_taxonomy["paraphrase_task"].astype(str).tolist()
# )
# np.save(
#     DATA_DIR / "taxonomy" / "per_llm" / "deepseek_embeddings_300.npy",
#     embeddings_deepseek,
# )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 671.99it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 8/8 [00:16<00:00,  2.12s/it]


In [14]:
from src.taxonomy.embedding import compute_embeddings_instruct

embeddings_mistral_instruct = compute_embeddings_instruct(
    mistral_taxonomy["paraphrase_task"].astype(str).tolist()
)
np.save(
    DATA_DIR / "taxonomy" / "per_llm" / "mistral_embeddings_instruct_300.npy",
    embeddings_mistral_instruct,
)


c:\Users\samma\Documents\Programmation\UdeM\nlp-benchmark-taxonomy\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\samma\.cache\huggingface\hub\models--intfloat--multilingual-e5-large-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 16/16 [15:20<00:00, 57.54s/it]

## 2. HAC Construction

### Construction

In [15]:
from src.taxonomy.hac_utils import (
    EXCLUDED_LABELS,
    build_linkage,
    clustering_metrics,
    cophenetic_table,
    dendrogram_purity,
    make_color_map,
    plot_dendrogram,
)

# ── Linkage matrices ──────────────────────────────────────────────────────────
embeddings_arr_m = np.array(embeddings_mistral_instruct)
# embeddings_arr_d = np.array(embeddings_deepseek)

Z_m = build_linkage(embeddings_arr_m)
# Z_d = build_linkage(embeddings_arr_d)

In [16]:
# Palette partagée (mêmes couleurs pour les deux dendrogrammes)
coarse_values_m = mistral_taxonomy["openai_coarse_task"].fillna("Unknown")
coarse_values_d = deepseek_taxonomy["openai_coarse_task"].fillna("Unknown")

color_map = make_color_map(pd.concat([coarse_values_m, coarse_values_d]))

In [18]:
# Dendrogramme Mistral
fig_m = plot_dendrogram(
    embeddings_arr_m,
    mistral_taxonomy,
    coarse_col="openai_coarse_task",
    label_col="openai_coarse_task",
    title="HAC — Mistral (coloré par coarse label)",
    color_map=color_map,
)
# fig_m.show(config={"responsive": True, "scrollZoom": True, "displayModeBar": True})

In [ ]:
# # Dendrogramme DeepSeek
# fig_d = plot_dendrogram(
#     embeddings_arr_d,
#     deepseek_taxonomy,
#     coarse_col="openai_coarse_task",
#     label_col="openai_coarse_task",
#     title="HAC — DeepSeek (coloré par coarse label)",
#     color_map=color_map,
# )
# fig_d.show(config={"responsive": True, "scrollZoom": True, "displayModeBar": True})


### Metrics

In [ ]:
# N_CLUSTERS = 20

# metrics_m = clustering_metrics(
#     embeddings_arr_m, Z_m, coarse_values_m, n_clusters=N_CLUSTERS
# )
# metrics_d = clustering_metrics(
#     embeddings_arr_d, Z_d, coarse_values_d, n_clusters=N_CLUSTERS
# )

# rows = {
#     "n_clusters": [metrics_m["n_clusters"], metrics_d["n_clusters"]],
#     "ARI vs coarse": [f"{metrics_m['ari']:.3f}", f"{metrics_d['ari']:.3f}"],
#     "NMI vs coarse": [f"{metrics_m['nmi']:.3f}", f"{metrics_d['nmi']:.3f}"],
#     "Silhouette": [f"{metrics_m['silhouette']:.3f}", f"{metrics_d['silhouette']:.3f}"],
#     "Cosine intra": [
#         f"{metrics_m['cosine_intra']:.3f}",
#         f"{metrics_d['cosine_intra']:.3f}",
#     ],
#     "Cosine inter": [
#         f"{metrics_m['cosine_inter']:.3f}",
#         f"{metrics_d['cosine_inter']:.3f}",
#     ],
#     "Ratio i/e": [
#         f"{metrics_m['cosine_ratio']:.3f}",
#         f"{metrics_d['cosine_ratio']:.3f}",
#     ],
# }
# summary = pd.DataFrame(rows, index=["Mistral", "DeepSeek"]).T
# print(f"k={N_CLUSTERS} clusters (maxclust)\n")
# print(summary.to_string())

# # ARI entre les deux clusterings
# ari_cross = adjusted_rand_score(metrics_m["hac_labels"], metrics_d["hac_labels"])
# print(f"\nARI Mistral ↔ DeepSeek (k={N_CLUSTERS}) : {ari_cross:.3f}")


In [ ]:
# # On compare les deux arbres contre les coarse labels

# dp_m = dendrogram_purity(Z_m, coarse_values_m.to_numpy())
# dp_d = dendrogram_purity(Z_d, coarse_values_d.to_numpy())

# n_classes = len([c for c in coarse_values_m.unique() if c not in EXCLUDED_LABELS])
# baseline = 1 / n_classes

# print(f"{'':30} {'Mistral':>10}  {'DeepSeek':>10}  {'baseline':>10}")
# print("─" * 64)
# print(f"{'Dendrogram Purity':<30} {dp_m:>10.3f}  {dp_d:>10.3f}  {baseline:>10.3f}")

# # ── Cophenetic intra / inter ──────────────────────────────────────────────────
# ct_m = cophenetic_table(Z_m, coarse_values_m.to_numpy())
# ct_d = cophenetic_table(Z_d, coarse_values_d.to_numpy())

# merged = ct_m.merge(ct_d, on="family", suffixes=("_mistral", "_deepseek"))
# merged = merged.sort_values("ratio_mistral")

# print("\nCophenetic intra / inter par famille :")
# print(
#     f"{'Famille':<30} {'intra_M':>8} {'inter_M':>8} {'ratio_M':>8}  {'intra_D':>8} {'inter_D':>8} {'ratio_D':>8}  {'n':>4}"
# )
# print("─" * 90)
# for _, row in merged.iterrows():
#     print(
#         f"{row['family']:<30}"
#         f" {row['intra_mistral']:>8.3f} {row['inter_mistral']:>8.3f} {row['ratio_mistral']:>8.3f}"
#         f"  {row['intra_deepseek']:>8.3f} {row['inter_deepseek']:>8.3f} {row['ratio_deepseek']:>8.3f}"
#         f"  {row['n_mistral']:>4.0f}"
#     )
# print("─" * 90)
# g_m = ct_m["intra"].mean() / ct_m["inter"].mean()
# g_d = ct_d["intra"].mean() / ct_d["inter"].mean()
# print(f"{'GLOBAL':<30} {'':>8} {'':>8} {g_m:>8.3f}  {'':>8} {'':>8} {g_d:>8.3f}")
# print("\nratio < 1 → famille bien regroupée  |  ratio > 1 → famille dispersée")


In [ ]:
# from scipy.cluster.hierarchy import cophenet, fcluster
# from scipy.stats import pearsonr, spearmanr

# # ── 1. Corrélation cophénétique croisée ───────────────────────────────────────
# coph_m = cophenet(Z_m)
# coph_d = cophenet(Z_d)

# r_pearson, p_pearson = pearsonr(coph_m, coph_d)
# r_spearman, p_spearman = spearmanr(coph_m, coph_d)

# print("=== Corrélation cophénétique croisée (Mistral ↔ DeepSeek) ===")
# print(f"Pearson  r = {r_pearson:.3f}   (p={p_pearson:.2e})")
# print(f"Spearman r = {r_spearman:.3f}   (p={p_spearman:.2e})")
# print("(1 = arbres identiques, 0 = aucune corrélation)")

# # ── 2. Dendrogram Purity croisé ───────────────────────────────────────────────
# # Labels issus de l'arbre de l'autre LLM → mesure si les deux arbres
# # regroupent les mêmes papiers ensemble.
# labels_m_k = fcluster(Z_m, t=N_CLUSTERS, criterion="maxclust").astype(str)
# labels_d_k = fcluster(Z_d, t=N_CLUSTERS, criterion="maxclust").astype(str)

# dp_m_on_d = dendrogram_purity(Z_d, labels_m_k, excluded=frozenset())
# dp_d_on_m = dendrogram_purity(Z_m, labels_d_k, excluded=frozenset())

# # Baseline par permutation : DP croisé attendu sous labels aléatoires
# rng = np.random.default_rng(42)
# N_SHUFFLE = 50
# baseline_m_on_d = np.mean(
#     [
#         dendrogram_purity(Z_d, rng.permutation(labels_m_k), excluded=frozenset())
#         for _ in range(N_SHUFFLE)
#     ]
# )
# baseline_d_on_m = np.mean(
#     [
#         dendrogram_purity(Z_m, rng.permutation(labels_d_k), excluded=frozenset())
#         for _ in range(N_SHUFFLE)
#     ]
# )

# print(
#     f"\n=== Dendrogram Purity croisé (k={N_CLUSTERS} clusters, {N_SHUFFLE} shuffles) ==="
# )
# print(f"{'':40} {'M→D':>8}  {'D→M':>8}")
# print("─" * 58)
# print(f"{"DP croisé (labels de l'autre LLM)":<40} {dp_m_on_d:>8.3f}  {dp_d_on_m:>8.3f}")
# print(
#     f"{'Baseline shuffle (aléatoire)':<40} {baseline_m_on_d:>8.3f}  {baseline_d_on_m:>8.3f}"
# )
# print(
#     f"{'Gain vs baseline':<40} {dp_m_on_d - baseline_m_on_d:>+8.3f}  {dp_d_on_m - baseline_d_on_m:>+8.3f}"
# )
# print()
# print("gain > 0 → les deux arbres capturent des structures similaires")
# print("gain ≈ 0 → les regroupements sont indépendants")


## 3. Clustering HAC Cutting

In [19]:
from src.taxonomy.hac_utils import (
    cut_tree_adaptive,
    plot_taxo_tree,
)

# -- Adaptive cut on Mistral embeddings --
taxo_tree_m = cut_tree_adaptive(Z_m, embeddings_arr_m, min_gain=0.015)

leaf_nodes = taxo_tree_m._leaf_nodes()
print(f"Clusters: {len(leaf_nodes)}")
print(f"Sizes: {sorted([len(n.leaves) for n in leaf_nodes], reverse=True)}")

# -- Visualisation --
fig_tree_m = plot_taxo_tree(
    taxo_tree_m,
    mistral_taxonomy.reset_index(drop=True),
    label_col="google_task_type",
    title="Adaptive HAC Taxonomy -- Mistral",
)

Clusters: 17
Sizes: [45, 35, 27, 24, 22, 18, 13, 13, 11, 10, 10, 6, 5, 4, 2, 1, 1]


In [20]:
# Attach cluster assignments to mistral_taxonomy and save
flat = taxo_tree_m.flat_labels(n=len(mistral_taxonomy))

leaf_nodes = taxo_tree_m._leaf_nodes()
cluster_label_map = {
    cid: df_row["openai_coarse_task"].value_counts().index[0]
    for cid, node in enumerate(leaf_nodes)
    for df_row in [mistral_taxonomy.reset_index(drop=True).iloc[node.leaves]]
}

mistral_taxonomy = mistral_taxonomy.reset_index(drop=True).copy()
mistral_taxonomy["hac_cluster_id"] = flat
mistral_taxonomy["hac_cluster_label"] = mistral_taxonomy["hac_cluster_id"].map(
    cluster_label_map
)

print(
    mistral_taxonomy[
        [
            "hac_cluster_id",
            # "hac_cluster_label"
        ]
    ]
    .value_counts()
    .head(20)
)

clustering_df = mistral_taxonomy[
    [
        "bibkey",
        "paraphrase_task",
        "google_task_type",
        "judgment_type",
        "hac_cluster_id",
        "hac_cluster_label",
        "title",
        "abstract",
    ]
]

hac_cluster_id
14                45
4                 35
16                27
3                 24
11                22
15                18
2                 13
5                 13
10                11
13                10
1                 10
7                  6
0                  5
9                  4
12                 2
8                  1
6                  1
Name: count, dtype: int64


In [ ]:
from scipy.cluster.hierarchy import linkage


def collect_leaves(node):
    if not node.children:
        return [node]
    out = []
    for c in node.children:
        out.extend(collect_leaves(c))
    return out


idx = clustering_df.index[clustering_df["hac_cluster_id"] == 14].tolist()
sub_Z = linkage(embeddings_arr_m[idx], method="average", metric="cosine")
root = cut_tree_adaptive(
    sub_Z, embeddings_arr_m[idx], max_size=6, tau=0.85, min_cluster_size=2
)

for leaf in collect_leaves(root):
    print(f"\n--- n={len(leaf.leaves)}, coh={leaf.coherence:.2f} ---")
    for j in leaf.leaves:
        print(f"  • {clustering_df.iloc[idx[j]]['paraphrase_task'][:140]}")